# Load test Pha 1 — Bank Account

Đo ba chức năng: đăng nhập, xem tài khoản và rút tiền đồng thời.
API, PostgreSQL 15 và k6 cùng chạy trên Kaggle CPU. Chọn **Accelerator: None**, **Internet: On**.

Chạy thử với `MODE = 'smoke'` trước. Sau đó đặt `MODE = 'full'` trong ô mới và chạy lại phần 9–11.
Kết quả chính thức có 27 lượt: 3 chức năng × 3 mức tải × 3 lần lặp.

## 1. Khai báo
Đổi REPO_URL và COMMIT nếu cần kiểm thử phiên bản source khác.

In [ ]:
import os
import json
import time
import uuid
import shutil
import secrets
import subprocess
from pathlib import Path

MODE = 'smoke'
REPO_URL = 'https://github.com/HUY-TD16/SA_BankAccount.git'
COMMIT = 'a4a0f831054dc13c3e366952cc960cecd941efe0'

ROOT = Path('/tmp/bank-source') / COMMIT
OUT = Path('/kaggle/working') / ('bank-test-' + time.strftime('%Y%m%d-%H%M%S'))
OUT.mkdir(parents=True, exist_ok=True)
BASE = 'http://127.0.0.1:3000/api/v1'
DB_URL = 'postgresql://postgres@127.0.0.1:55432/bank_load_test'

def run(command, cwd=None):
    return subprocess.run(command, cwd=cwd, check=True, text=True)

def read(command):
    return subprocess.check_output(command, text=True).strip()


Các thông số đo để ở đây cho dễ đọc. Mỗi client nghỉ 0,1 giây sau một request.

In [ ]:
settings = {
    "id": "sa-bank-phase1-kaggle-v1",
    "workloads": [
        "login",
        "list",
        "debit"
    ],
    "vus": [
        1,
        5,
        20
    ],
    "repeats": 3,
    "duration": "60s",
    "warmup": "10s",
    "cooldown_seconds": 2,
    "think_time_seconds": 0.1,
    "request_timeout": "30s",
    "graceful_stop": "35s",
    "fixture": {
        "users": 1,
        "accounts": 20,
        "account_type": "PAYMENT",
        "currency": "VND",
        "initial_balance_per_account": "1000000000",
        "debit_amount": "1",
        "list_page": 1,
        "list_limit": 20,
        "reset_before_each_run": True
    },
    "error_threshold": "rate<0.01"
}

Path('/tmp/bank-protocol.json').write_text(json.dumps(settings))
(OUT / 'protocol.json').write_text(json.dumps(settings, indent=2))


## 2. Cài công cụ
Dùng Node 24.22.0, PostgreSQL 15 và k6 1.3.0. PostgreSQL được cài từ kho PGDG; phiên bản thực tế được lưu trong environment.json. Lần đo ngày 28/10/2026 (UTC) dùng PostgreSQL 15.19. psutil dùng để đọc CPU/RAM của API theo PID.

In [ ]:
%%bash
set -e
apt-get update -qq
apt-get install -y -qq git curl xz-utils ca-certificates openssl

. /etc/os-release
mkdir -p /usr/share/postgresql-common/pgdg
curl -fL --retry 3 https://www.postgresql.org/media/keys/ACCC4CF8.asc -o /usr/share/postgresql-common/pgdg/apt.postgresql.org.asc
echo "deb [signed-by=/usr/share/postgresql-common/pgdg/apt.postgresql.org.asc] https://apt.postgresql.org/pub/repos/apt ${VERSION_CODENAME}-pgdg main" > /etc/apt/sources.list.d/pgdg.list
apt-get update -qq
apt-get install -y -qq postgresql-15 postgresql-client-15
python -m pip install -q psutil==7.2.2

mkdir -p /tmp/bank-tools
cd /tmp/bank-tools
curl -fL --retry 3 https://nodejs.org/dist/v24.22.0/node-v24.22.0-linux-x64.tar.xz -o node.tar.xz
tar -xf node.tar.xz
curl -fL --retry 3 https://github.com/grafana/k6/releases/download/v1.3.0/k6-v1.3.0-linux-amd64.tar.gz -o k6.tar.gz
tar -xzf k6.tar.gz


## 3. Lấy source và build
Phải cài cả devDependencies vì lệnh build cần Nest CLI.

In [ ]:
import requests
import psutil
import pandas as pd

os.environ['PATH'] = '/tmp/bank-tools/node-v24.22.0-linux-x64/bin:/tmp/bank-tools/k6-v1.3.0-linux-amd64:' + os.environ['PATH']
os.environ.update({
    'DATABASE_URL': DB_URL, 'NODE_ENV': 'production', 'PORT': '3000',
    'JWT_ACCESS_SECRET': secrets.token_hex(32), 'JWT_ACCESS_TTL': '2h',
    'BCRYPT_ROUNDS': '12', 'UV_THREADPOOL_SIZE': '4',
})
if not ROOT.exists():
    run(['git', 'clone', REPO_URL, str(ROOT)])
run(['git', 'checkout', '--detach', COMMIT], ROOT)
assert read(['git', '-C', str(ROOT), 'rev-parse', 'HEAD']) == COMMIT
run(['npm', 'ci', '--include=dev'], ROOT)
run(['npx', '--no-install', 'prisma', 'generate'], ROOT)
run(['npm', 'run', 'build'], ROOT)
shutil.copy(ROOT / 'package-lock.json', OUT)
run(['git', 'archive', '--format=tar', '--output=' + str(OUT / 'source.tar'), COMMIT], ROOT)


## 4. Tạo database test
Database `bank_load_test` chỉ dành cho bài này. Các lượt đo sẽ xóa dữ liệu test rồi tạo lại.

In [ ]:
PG_BIN = '/usr/lib/postgresql/15/bin/'
PG_DATA = '/tmp/bank-postgres-15'
Path(PG_DATA).mkdir(exist_ok=True)
run(['chown', 'postgres:postgres', PG_DATA])
if not Path(PG_DATA, 'PG_VERSION').exists():
    run(['runuser', '-u', 'postgres', '--', PG_BIN + 'initdb', '-D', PG_DATA,
         '-A', 'trust', '--encoding=UTF8', '--locale=C'])
status = subprocess.run(['runuser', '-u', 'postgres', '--', PG_BIN + 'pg_ctl', '-D', PG_DATA, 'status'], capture_output=True)
if status.returncode != 0:
    run(['runuser', '-u', 'postgres', '--', PG_BIN + 'pg_ctl', '-D', PG_DATA,
         '-l', PG_DATA + '/server.log', '-o', '-h 127.0.0.1 -p 55432', '-w', 'start'])

def sql(query, database=DB_URL):
    return read([PG_BIN + 'psql', database, '-v', 'ON_ERROR_STOP=1', '-Atc', query])

admin = 'postgresql://postgres@127.0.0.1:55432/postgres'
assert 150000 <= int(sql('SHOW server_version_num', admin)) < 160000, 'Expected PostgreSQL 15; start a new Kaggle session'
if sql("SELECT 1 FROM pg_database WHERE datname='bank_load_test'", admin) != '1':
    sql('CREATE DATABASE bank_load_test', admin)
run(['npx', '--no-install', 'prisma', 'migrate', 'deploy'], ROOT)
print('Database ready:', sql('SHOW server_version'))


## 5. Chạy API
Request chưa có token trả 401 là đúng. Nếu app không chạy được thì xem `app.log`.

In [ ]:
if 'app' in globals() and app.poll() is None:
    app.terminate()
    app.wait(timeout=15)
app_log = open(OUT / 'app.log', 'w')
app = subprocess.Popen(['node', 'dist/main.js'], cwd=ROOT,
                       stdout=app_log, stderr=subprocess.STDOUT)
ready = False
for _ in range(60):
    if app.poll() is not None:
        break
    try:
        ready = requests.get(BASE + '/accounts', timeout=1).status_code == 401
        if ready:
            break
    except requests.RequestException:
        pass
    time.sleep(1)
assert ready, (OUT / 'app.log').read_text()[-4000:]
print('API ready, PID:', app.pid)


## 6. Ghi lại môi trường
Lưu cấu hình thực tế để biết số liệu được đo trên máy và phiên bản nào.

In [ ]:
environment = {
    'commit': COMMIT, 'repo': REPO_URL,
    'cpu': read(['lscpu']), 'memory': read(['free', '-b']),
    'node': read(['node', '--version']), 'k6': read(['k6', 'version']),
    'postgres': sql('SELECT version()'), 'psutil': psutil.__version__,
    'bcrypt_rounds': 12, 'uv_threadpool_size': 4,
    'topology': 'API, PostgreSQL and k6 on the same Kaggle CPU session',
    'postgres_settings': sql("SELECT name || '=' || setting FROM pg_settings WHERE name IN ('max_connections','shared_buffers','work_mem','fsync','synchronous_commit') ORDER BY name"),
}
for name in ['cpu.max', 'cpuset.cpus.effective', 'memory.max']:
    file = Path('/sys/fs/cgroup') / name
    environment[name] = file.read_text().strip() if file.exists() else 'unavailable'
(OUT / 'environment.json').write_text(json.dumps(environment, indent=2))
print(json.dumps(environment, indent=2))


## 7. Script k6
Đây là code gửi request. Login và list dùng HTTP 200; debit dùng HTTP 201.
Các client debit cùng account đầu tiên, mỗi lần rút 1 VND.

In [ ]:
%%writefile /tmp/bank-load-k6.js
import http from 'k6/http';
import { sleep } from 'k6';
import { Counter, Rate, Trend } from 'k6/metrics';

// Fixtures are prepared outside k6: setup traffic is excluded from these metrics.
const fixture = JSON.parse(open(__ENV.FIXTURE));
const protocol = JSON.parse(open(__ENV.PROTOCOL_FILE));
const workload = __ENV.WORKLOAD || 'list';
if (!protocol.workloads.includes(workload)) throw new Error('Invalid WORKLOAD');
const isDebit = workload === 'debit';
const vus = Number(__ENV.VUS || 1);
const latency = new Trend('operation_duration', true);
const errors = new Rate('operation_errors');
const requests = new Counter('operation_requests');
const successes = new Counter('operation_successes');
const base = 'http://127.0.0.1:3000/api/v1';
const thresholds = { operation_errors: [protocol.error_threshold] };

export const options = {
  scenarios: {
    measured: {
      executor: 'constant-vus', vus,
      duration: __ENV.DURATION || protocol.duration, gracefulStop: protocol.graceful_stop,
    },
  },
  summaryTrendStats: ['avg', 'min', 'med', 'p(90)', 'p(95)', 'p(99)', 'max'],
  thresholds,
};

export default function () {
  const params = {
    timeout: protocol.request_timeout,
    headers: { 'Content-Type': 'application/json', Authorization: `Bearer ${fixture.token}` },
    tags: { name: workload },
  };
  let response;
  // All clients debit the first account to exercise concurrent updates to one balance.
  const accountId = fixture.accountIds[0];
  if (workload === 'login') {
    response = http.post(`${base}/auth/login`, JSON.stringify({
      email: fixture.email, password: fixture.password,
    }), params);
  } else if (workload === 'list') {
    response = http.get(`${base}/accounts?page=${protocol.fixture.list_page}&limit=${protocol.fixture.list_limit}`, params);
  } else {
    response = http.post(`${base}/accounts/${accountId}/transactions`,
      JSON.stringify({ type: 'DEBIT', amount: protocol.fixture.debit_amount }), params);
  }
  let body;
  try { body = response.json(); } catch (_) { body = null; }
  let ok = response.status === (isDebit ? 201 : 200) && body?.success === true;
  if (workload === 'login') ok = ok && typeof body?.data?.accessToken === 'string';
  if (workload === 'list') {
    const ids = Array.isArray(body?.data) ? body.data.map((account) => account?.id) : null;
    ok = ok && Array.isArray(ids) && ids.length === fixture.accountIds.length &&
      new Set(ids).size === fixture.accountIds.length && ids.every((id) => fixture.accountIds.includes(id));
  }
  if (isDebit) {
    ok = ok && body?.data?.amount === protocol.fixture.debit_amount && body?.data?.transactionType === 'DEBIT' && body?.data?.accountId === accountId;
  }
  latency.add(response.timings.duration);
  errors.add(!ok);
  requests.add(1);
  successes.add(ok ? 1 : 0);
  sleep(protocol.think_time_seconds); // Fixed think time; VUs are not requests/second.
}

export function handleSummary(data) {
  return {
    [__ENV.SUMMARY]: JSON.stringify(data, null, 2),
    stdout: JSON.stringify({ workload, metrics: {
      latency: data.metrics.operation_duration?.values,
      errors: data.metrics.operation_errors?.values,
      requests: data.metrics.operation_requests?.values,
    } }, null, 2) + '\n',
  };
}


In [ ]:
shutil.copy('/tmp/bank-load-k6.js', OUT / 'kaggle-load.js')


## 8. Chuẩn bị dữ liệu và hàm chạy test
Mỗi lượt có 1 user, 20 account; mỗi account ban đầu có 1 tỷ VND.

In [ ]:
PASSWORD = 'KaggleLoad123!'
INITIAL = int(settings['fixture']['initial_balance_per_account'])

def api(method, path, token=None, body=None, status=200):
    headers = {'Authorization': 'Bearer ' + token} if token else {}
    response = requests.request(method, BASE + path, headers=headers, json=body, timeout=35)
    assert response.status_code == status, response.text
    data = response.json()
    assert data['success'] is True, data
    return data['data']

def create_data():
    assert DB_URL == 'postgresql://postgres@127.0.0.1:55432/bank_load_test'
    sql('TRUNCATE financial_transactions, transfers, bank_accounts, users')
    email = uuid.uuid4().hex + '@example.test'
    api('POST', '/auth/register', body={'fullName': 'Load Test', 'email': email,
        'password': PASSWORD, 'confirmPassword': PASSWORD}, status=201)
    token = api('POST', '/auth/login', body={'email': email, 'password': PASSWORD})['accessToken']
    accounts = []
    for i in range(settings['fixture']['accounts']):
        account = api('POST', '/accounts', token,
                      {'accountName': 'Account ' + str(i), 'accountType': 'PAYMENT',
                       'initialDeposit': str(INITIAL)}, status=201)
        accounts.append(account['id'])
    data = {'email': email, 'password': PASSWORD, 'token': token, 'accountIds': accounts}
    Path('/tmp/bank-data.json').write_text(json.dumps(data))
    return data


Hàm này chạy một lượt và lưu CPU/RAM API mỗi giây bằng PID.
Sau tải, đối chiếu số dư và số bút toán của từng account.

In [ ]:
def run_test(workload, vus, duration, label):
    assert app.poll() is None, 'API has stopped'
    data = create_data()
    (OUT / (label + '-input.json')).write_text(json.dumps({
        'fixture': settings['fixture'], 'account_ids': data['accountIds'],
        'workload': workload, 'vus': vus, 'duration': duration}, indent=2))
    env = dict(os.environ, FIXTURE='/tmp/bank-data.json', PROTOCOL_FILE='/tmp/bank-protocol.json',
               WORKLOAD=workload, VUS=str(vus), DURATION=duration,
               SUMMARY=str(OUT / (label + '.json')))
    usage = []
    process = psutil.Process(app.pid)
    with open(OUT / (label + '.log'), 'w') as log:
        job = subprocess.Popen(['k6', 'run', '--quiet', '--no-usage-report', '/tmp/bank-load-k6.js'],
                               env=env, stdout=log, stderr=subprocess.STDOUT)
        try:
            while job.poll() is None:
                cpu = process.cpu_percent(interval=1)
                usage.append({'time': time.time(), 'pid': app.pid, 'cpu_pct': cpu,
                              'rss_mib': process.memory_info().rss / 1024**2})
        finally:
            if job.poll() is None:
                job.terminate()
            job.wait(timeout=10)
    pd.DataFrame(usage).to_csv(OUT / (label + '-cpu.csv'), index=False)
    assert job.returncode in (0, 99), (OUT / (label + '.log')).read_text()[-3000:]
    assert len(usage) >= 2, 'Missing API resource samples'

    metrics = json.loads((OUT / (label + '.json')).read_text())['metrics']
    latency = metrics['operation_duration']['values']
    success = metrics['operation_successes']['values']['count']
    balances = []
    for i, account_id in enumerate(data['accountIds']):
        expected = success if workload == 'debit' and i == 0 else 0
        balance = int(api('GET', '/accounts/' + account_id, data['token'])['currentBalance'])
        account_id = str(uuid.UUID(account_id))
        entries = int(sql("SELECT count(*) FROM financial_transactions WHERE account_id='" + account_id + "' AND transaction_type='DEBIT'"))
        balances.append({'account_id': account_id, 'initial': INITIAL, 'final': balance,
                         'successes': expected, 'debit_entries': entries,
                         'ok': balance == INITIAL - expected and entries == expected and balance >= 0})
    (OUT / (label + '-balance.json')).write_text(json.dumps(balances, indent=2))
    row = {'workload': workload, 'vus': vus, 'duration': duration,
           'requests': metrics['operation_requests']['values']['count'],
           'rps': metrics['operation_requests']['values']['rate'],
           'p50_ms': latency['med'], 'p95_ms': latency['p(95)'], 'p99_ms': latency['p(99)'],
           'error_pct': metrics['operation_errors']['values']['rate'] * 100,
           'api_cpu_avg_pct': sum(s['cpu_pct'] for s in usage) / len(usage),
           'api_rss_max_mib': max(s['rss_mib'] for s in usage),
           'balance_ok': all(b['ok'] for b in balances)}
    print(label, row, flush=True)
    assert row['balance_ok'], 'Balance check failed; inspect *-balance.json'
    return row


## 9. Chạy tải
Smoke: mỗi chức năng 5 giây. Full: mỗi lượt 60 giây, warm-up riêng 10 giây, lặp 3 lần.
Muốn đo chính thức, chạy ô `MODE = 'full'` ở đây rồi chạy lại phần 9–11.

In [ ]:
assert MODE in ['smoke', 'full']
if MODE == 'smoke':
    cases = [('login', 1), ('list', 1), ('debit', 2)]
    repeats, duration = 1, '5s'
else:
    cases = [(w, v) for w in settings['workloads'] for v in settings['vus']]
    repeats, duration = settings['repeats'], settings['duration']

plan = {'mode': MODE, 'cases': cases, 'repeats': repeats, 'duration': duration,
        'warmup': settings['warmup'] if MODE == 'full' else None,
        'think_time_seconds': settings['think_time_seconds'], 'reverse_even_repeat': True}
(OUT / ('plan-' + MODE + '.json')).write_text(json.dumps(plan, indent=2))
rows = []
for repeat in range(1, repeats + 1):
    order = cases if repeat % 2 else list(reversed(cases))
    for workload, vus in order:
        label = f'{MODE}-{workload}-{vus}-run{repeat}'
        if MODE == 'full':
            run_test(workload, vus, settings['warmup'], label + '-warmup')
            time.sleep(settings['cooldown_seconds'])
        row = run_test(workload, vus, duration, label)
        row['repeat'] = repeat
        rows.append(row)
        pd.DataFrame(rows).to_csv(OUT / ('results-' + MODE + '.csv'), index=False)
        time.sleep(settings['cooldown_seconds'])
display(pd.DataFrame(rows))


## 10. Thử rút vượt số dư
Cho 20 request cùng rút 10.000 VND từ account có 100.000 VND.
Mong đợi 10 lần thành công, 10 lần thiếu tiền, số dư 0 và 10 bút toán DEBIT.
Bài này chạy riêng, không cộng các response 409 vào error rate ở trên.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from threading import Barrier

data = create_data()
account = api('POST', '/accounts', data['token'],
              {'accountName': 'Race test', 'accountType': 'PAYMENT', 'initialDeposit': '100000'}, status=201)
barrier = Barrier(20)

def withdraw(_):
    barrier.wait(timeout=30)
    response = requests.post(BASE + '/accounts/' + account['id'] + '/transactions',
                             headers={'Authorization': 'Bearer ' + data['token']},
                             json={'type': 'DEBIT', 'amount': '10000'}, timeout=35)
    return response.status_code, response.json()

with ThreadPoolExecutor(max_workers=20) as pool:
    responses = list(pool.map(withdraw, range(20)))
success = sum(status == 201 and body.get('success') is True for status, body in responses)
rejected = sum(status == 409 and body.get('error', {}).get('code') == 'INSUFFICIENT_FUNDS' for status, body in responses)
balance = api('GET', '/accounts/' + account['id'], data['token'])['currentBalance']
account_id = str(uuid.UUID(account['id']))
entries = int(sql("SELECT count(*) FROM financial_transactions WHERE account_id='" + account_id + "' AND transaction_type='DEBIT'"))
race_result = {'success_201': success, 'insufficient_409': rejected, 'final_balance': balance, 'debit_entries': entries}
(OUT / ('race-' + MODE + '.json')).write_text(json.dumps(race_result, indent=2))
print(race_result)
assert (success, rejected, balance, entries) == (10, 10, '0', 10), 'Overdraft check failed'


## 11. Xem bảng kết quả và tải ZIP
Lấy median từng chỉ số qua ba lượt; đây không phải percentile của dữ liệu gộp.
CPU 100% tương đương một CPU logic; RSS tính bằng MiB. Error threshold <1% chỉ là mốc thử nghiệm.
ZIP giữ cả số liệu từng lượt, cấu hình máy, dữ liệu đầu vào, log và kiểm tra số dư.

In [ ]:
df = pd.DataFrame(rows)
summary = df.groupby(['workload', 'vus'])[['rps', 'p50_ms', 'p95_ms', 'p99_ms',
    'error_pct', 'api_cpu_avg_pct', 'api_rss_max_mib']].median().round(2)
display(summary)
summary.to_csv(OUT / ('summary-' + MODE + '.csv'))

report = {'mode': MODE, 'commit': COMMIT, 'expected_runs': len(cases) * repeats,
          'completed_runs': len(rows), 'balance_ok': all(r['balance_ok'] for r in rows),
          'error_threshold_pass': all(r['error_pct'] < 1 for r in rows),
          'overdraft_check': race_result}
(OUT / ('report-' + MODE + '.json')).write_text(json.dumps(report, indent=2))
archive = shutil.make_archive(str(OUT) + '-' + MODE, 'zip', OUT)
from IPython.display import FileLink
display(FileLink(archive))
print('Saved:', archive)


Nếu có lỗi khởi động, xem log bằng ô dưới. Không dùng kết quả smoke làm báo cáo hiệu năng.

In [ ]:
print((OUT / 'app.log').read_text()[-4000:])
